<a href="https://colab.research.google.com/github/asaveraasad-data/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import os
import subprocess

REPO_URL = "https://github.com/asaveraasad-data/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
       subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)
print("Current directory:", os.getcwd())

Current directory: /content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


# Signal Checks

Before building my baseline rule, I checked whether the signals I want to use are actually supported by the data.

## Signal Check 1
**Question:** Do older (stale) pages show a higher proportion of declining performance?

**Verdict:** MIXED

Older pages (365+ days) have the highest proportion of declining pages (60%), but this bucket contains only 5 pages, making it too small for a reliable conclusion. In contrast, recently updated pages (0–180 days) also show a high decline rate (54.2%). This suggests that staleness alone is not sufficient for prioritizing content refreshes and should be combined with other signals.

---

## Signal Check 2
**Question:** Does CTR decrease as search position becomes worse?

**Verdict:** CONFIRMED

The average CTR consistently decreases as search position becomes worse, from 1.48% in the top_3 tier to 0.15% in the deep tier. This supports using CTR and search position together when identifying pages that may benefit from content review.

In [14]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Bucket pages by days since last update
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 180, 365, 10000],
    labels=["0-180", "181-365", "365+"]
)

# Percentage of pages in each trend direction
signal1 = (
    df.groupby("staleness_bucket")["trend_direction"]
      .value_counts(normalize=True)
      .unstack(fill_value=0)
      .round(3)
)

# Sample size
signal1["n"] = df.groupby("staleness_bucket").size()

print(signal1)

trend_direction    down   flat    new  stable     up      n
staleness_bucket                                           
0-180             0.542  0.038  0.074   0.199  0.146  29826
181-365           0.467  0.089  0.142   0.142  0.160    169
365+              0.600  0.200  0.200   0.000  0.000      5


/tmp/ipykernel_695/2835085475.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("staleness_bucket")["trend_direction"]
/tmp/ipykernel_695/2835085475.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1["n"] = df.groupby("staleness_bucket").size()


In [15]:
signal2 = (
    df.groupby("position_tier")
      .agg(
          mean_ctr=("ctr","mean"),
          n=("ctr","size")
      )
      .sort_values("mean_ctr", ascending=False)
)

print(signal2.round(4))

               mean_ctr      n
position_tier                 
top_3            1.4836   2321
page_1           0.6525  11814
striking         0.3232   7304
page_3_5         0.2225   7242
deep             0.1502   1319


## My Rule

A page should be prioritized for content review if it is old, receives a high number of impressions, and has a low click-through rate (CTR). These pages already have visibility but may be underperforming, making them good candidates for content refresh.

### Reason Code

STALE_HIGH_IMPRESSIONS_LOW_CTR

### Action Label

Review for Content Refresh

In [16]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# -----------------------------
# Rule Conditions
# -----------------------------

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
low_ctr = (df["ctr"] <= df["ctr"].median()).astype(int)

# -----------------------------
# Baseline Score
# -----------------------------

df["baseline_score"] = (
    stale * 2 +
    visible * 3 +
    low_ctr * 2
)

# -----------------------------
# Reason Code
# -----------------------------

df["reason_code"] = "STALE_HIGH_IMPRESSIONS_LOW_CTR"

# -----------------------------
# Action Label
# -----------------------------

df["action"] = "Review for Content Refresh"

# View highest-priority pages
df.sort_values("baseline_score", ascending=False).head(10)[
    [
        "baseline_score",
        "impressions_90d",
        "days_since_last_update",
        "ctr",
        "reason_code",
        "action"
    ]
]

,baseline_score,impressions_90d,days_since_last_update,ctr,reason_code,action
11489,7,7812,194,0.01,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
698,7,4590,194,0.00,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
3507,7,533,183,0.00,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
23538,5,2198,20,0.05,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
18656,5,3633,20,0.00,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
11062,5,1137,104,0.00,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
18663,5,947,104,0.00,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
18665,5,9239,104,0.03,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
23543,5,816,22,0.00,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh
5577,5,6363,104,0.00,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh


## 2. Build the ranked queue (writes the CSV)

Code the score, rank everything, and write
work/outputs/baseline_action_score.csv.

In [17]:
import os

# Create output folder if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

# Rank pages
ranked_queue = (
    df.sort_values("baseline_score", ascending=False)
      .reset_index(drop=True)
)

# Save only useful columns
output = ranked_queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "days_since_last_update",
        "ctr",
        "avg_position"
    ]
]

# Write CSV
output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")
print(output.head(10))

CSV saved successfully!
             content_id  baseline_score                     reason_code  \
0  content_5feee3994adb               7  STALE_HIGH_IMPRESSIONS_LOW_CTR   
1  content_b16bd7307b39               7  STALE_HIGH_IMPRESSIONS_LOW_CTR   
2  content_074ba6ead17b               7  STALE_HIGH_IMPRESSIONS_LOW_CTR   
3  content_ca9a0abda355               5  STALE_HIGH_IMPRESSIONS_LOW_CTR   
4  content_e2b8e97ea579               5  STALE_HIGH_IMPRESSIONS_LOW_CTR   
5  content_c5e63875b21a               5  STALE_HIGH_IMPRESSIONS_LOW_CTR   
6  content_af796c8d8387               5  STALE_HIGH_IMPRESSIONS_LOW_CTR   
7  content_81e59177b59a               5  STALE_HIGH_IMPRESSIONS_LOW_CTR   
8  content_3ef9dba1efc9               5  STALE_HIGH_IMPRESSIONS_LOW_CTR   
9  content_98db1536110c               5  STALE_HIGH_IMPRESSIONS_LOW_CTR   

                       action  impressions_90d  days_since_last_update   ctr  \
0  Review for Content Refresh             7812                     194

## 3. Top-20 Review

The table below reviews the highest-ranked pages produced by the baseline rule.

For each page I include:
- Action
- Reason Code
- Confidence Note
- What would make the recommendation wrong

In [18]:
# Confidence based on the baseline score
top20["confidence_note"] = top20["baseline_score"].apply(
    lambda x: "High" if x >= 7 else "Medium"
)

# Explanation of what could make the recommendation wrong
top20["what_would_make_it_wrong"] = top20.apply(
    lambda row:
    "The page may have been updated recently or traffic could be seasonal."
    if row["baseline_score"] >= 7
    else
    "The page may have low CTR because of search intent rather than outdated content.",
    axis=1
)

top20[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

,content_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,content_5feee3994adb,7,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,High,The page may have been updated recently or tra...
1,content_b16bd7307b39,7,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,High,The page may have been updated recently or tra...
2,content_074ba6ead17b,7,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,High,The page may have been updated recently or tra...
3,content_ca9a0abda355,5,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,Medium,The page may have low CTR because of search in...
4,content_e2b8e97ea579,5,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,Medium,The page may have low CTR because of search in...
5,content_c5e63875b21a,5,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,Medium,The page may have low CTR because of search in...
6,content_af796c8d8387,5,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,Medium,The page may have low CTR because of search in...
7,content_81e59177b59a,5,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,Medium,The page may have low CTR because of search in...
8,content_3ef9dba1efc9,5,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,Medium,The page may have low CTR because of search in...
9,content_98db1536110c,5,STALE_HIGH_IMPRESSIONS_LOW_CTR,Review for Content Refresh,Medium,The page may have low CTR because of search in...


## 4. Weak picks + leakage check

### Weak picks

This baseline uses a simple rule based on observable signals: days since the last update, impressions, and CTR. Although these signals help identify pages that may need a content refresh, some recommendations may still be incorrect.

A page could receive a high score because of seasonal traffic changes, recent content updates, changes in search intent, or other external factors that are not available in this dataset. Therefore, this baseline should be treated as a decision-support tool rather than a final decision.

### Leakage check

To avoid data leakage, this baseline only uses information that would have been available before making a recommendation.

The following columns were **not** used because they are derived from the outcome or represent future information:

- `trend_direction`
- `trend_pct`
- `is_declining_label`

Excluding these columns ensures that the baseline remains honest and can later be compared fairly against machine learning models.

In [19]:
print("Leakage Check")

used_features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr"
]

print("Features used:")
for feature in used_features:
    print("-", feature)

print("\nExcluded columns (to avoid leakage):")
excluded = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

for feature in excluded:
    print("-", feature)

Leakage Check
Features used:
- days_since_last_update
- impressions_90d
- ctr

Excluded columns (to avoid leakage):
- trend_direction
- trend_pct
- is_declining_label


## Self-check

Before you submit, confirm each line honestly:

✅Every section above is filled — markdown thinking AND the code that backs it

✅The notebook runs top to bottom with no errors (Runtime → Run all)

✅No client names, URLs, or private queries anywhere

✅My claims use careful words: observed, measured, directional, decision-support

✅Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.